# import packages

In [161]:
import torch
import matplotlib.pyplot as plot

# 100个学生的8门学科成绩

In [162]:
# generate 8 classes
classes=["math","physics","chemical","literature","english","history","sports","geography"]

# number of classes and students
N_CLASS=len(classes)
N_STUDENTS=100

# generate grades table of students
grades=torch.zeros([N_STUDENTS,N_CLASS],dtype=torch.float32)

# generate detail grades
grades[:,0]=torch.normal(mean=75.0, std=15.0,size=(N_STUDENTS,))
grades[:,1]=grades[:,0]+torch.normal(mean=0.0, std=5.0,size=(N_STUDENTS,))
grades[:,2]=grades[:,0]+torch.normal(mean=0.0, std=5.0,size=(N_STUDENTS,))

grades[:,3]=torch.normal(mean=70.0, std=15.0,size=(N_STUDENTS,))
grades[:,4]=grades[:,3]+torch.normal(mean=0.0, std=10.0,size=(N_STUDENTS,))
grades[:,5]=grades[:,3]+torch.normal(mean=0.0, std=10.0,size=(N_STUDENTS,))

grades[:,6]=torch.normal(mean=80.0,std=5.0,size=(N_STUDENTS,))
grades[:,7]=grades[:,6]+torch.normal(mean=0.0,std=2.0,size=(N_STUDENTS,))

# clamp
grades=torch.clamp_(grades,min=0.0,max=100.0)

# show
# for grade in grades:
#     print(f"grade:{grade}")

# sigma

# mean
print(f'=======================================')
mean=torch.mean(grades,dim=0)
print(f'mean.shape:{mean.shape}, mean:{mean}')

# centered
grades_centered=grades-grades.mean(dim=0)

# sigma
grades_centered_T=grades_centered.T # shape (8,100)
sigma=grades_centered_T@grades_centered

# eigen vectors, eigen values by ascending
print(f'=======================================')
eigen_values, eigen_vectors=torch.linalg.eigh(sigma)
print(f'eigen_values:{eigen_values}, \neigen_vectors:\n{eigen_vectors}')

print(f'=======================================')
std_eigen_values=torch.sqrt(eigen_values)
print(f'std_eigen_values:{std_eigen_values}')

# descending
print(f'=======================================')
eigen_values, indices=torch.sort(eigen_values, descending=True) # 降序
eigen_vectors=eigen_vectors[:,indices] # 降序
std_eigen_values=std_eigen_values[indices] # 降序
print(f'eigen_values:{eigen_values}, \neigen_vectors:\n{eigen_vectors}')
print(f'std_eigen_values:{std_eigen_values}')

# analysis
print(f'=======================================')
eigen_value_sum=torch.sum(eigen_values)
print(f'eigen_value_sum:{eigen_value_sum}')
cumulative_percentage=0.0
for (i,eigen_value) in enumerate(eigen_values):
    percentage = eigen_value/eigen_value_sum*100
    cumulative_percentage+=percentage
    print(f'std eigen value index:{i}, percentage:{percentage:.2f}%, cumulative:{cumulative_percentage:.2f}%')

# PCA, use first 3 eigen vectors
print(f'=======================================')
PCA_3=eigen_vectors[:,:3] # shape (8,3)
print(f'PCA_3.shape:{PCA_3.shape}')
for c in range(3):
    print(f'PCA_{c}\n')
    for r in range(N_CLASS):
        if abs(PCA_3[r,c])>=0.1:
            print(f'\t{classes[r]}:{PCA_3[r,c]}')


# project to PCA 3, 每个学生在PCA-1/2/3这三个方向的能力量化
print(f'=======================================')
projected_grades=grades_centered@PCA_3
print(f'PCA_3:\n{PCA_3},\n projected_grades:\n{projected_grades}')





mean.shape:torch.Size([8]), mean:tensor([76.9574, 77.6803, 77.4623, 71.1112, 70.0044, 72.1082, 80.0838, 79.9476])
eigen_values:tensor([  244.7581,   770.0068,  1613.4751,  3391.2502,  4178.8755,  8277.0977,
        59009.2461, 67361.8359]), 
eigen_vectors:
tensor([[ 0.0554,  0.8106, -0.0628, -0.0425,  0.0186, -0.0355,  0.4146, -0.4009],
        [-0.0525, -0.3567,  0.7092, -0.1241, -0.1323, -0.0293,  0.4285, -0.3869],
        [-0.0073, -0.4566, -0.6629,  0.1205,  0.0800, -0.0326,  0.4111, -0.4013],
        [ 0.0089, -0.0364, -0.1475, -0.7893, -0.1205,  0.1430, -0.4005, -0.3980],
        [-0.0126,  0.0046,  0.0734,  0.2686,  0.0942, -0.7414, -0.4377, -0.4150],
        [-0.0169,  0.0246,  0.0884,  0.5071, -0.0466,  0.6423, -0.3514, -0.4427],
        [-0.7036,  0.0252,  0.0648, -0.0997,  0.6947,  0.0842, -0.0015, -0.0227],
        [ 0.7061, -0.0686,  0.1205, -0.0771,  0.6838,  0.0846, -0.0091, -0.0371]])
std_eigen_values:tensor([ 15.6447,  27.7490,  40.1681,  58.2344,  64.6442,  90.9786, 2